<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/AtomVisionExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Example to run [AtomVision](https://github.com/usnistgov/atomvision)

AtomVision is a deep learning package to perform various operations on image data including segmentation and classification of images.

**For faster computing, change notebook hardware to "GPU". Edit > Notebook Settings > Hardware Accelerator > GPU**

# Find more detail about AtomVision in: https://pubs.acs.org/doi/10.1021/acs.jcim.2c01533

# Table of Contents
1. Install [AtomVision](https://github.com/usnistgov/atomvision) package
2. Generating STEM image with convolution approximation: graphene example
3. Train an autoencoder
4. Train a Generative Adversarial Network
5. Train DenseNet classification model on JARVIS-DFT 2D STEM image dataset
6. Generate Graph over Atomistic Image
7. Train ALIGNN based classification model on JARVIS-DFT 2D STEM image dataset


# Get the repository and install

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# %%time

# !conda install alignn dgl=2.1.0 pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia --quiet -y


In [ ]:
!pip install  -q git+https://github.com/usnistgov/atomvision.git@develop

In [ ]:
# pip install Pillow==9.0.0

In [ ]:
!conda install ipykernel -q -y

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('atomvision'):
  !git clone https://github.com/usnistgov/atomvision.git
os.chdir('atomvision')

Generating STEM image with convolution approximation: graphene example

In [ ]:
!python atomvision/scripts/stem_conv.py --file_path atomvision/tests/POSCAR --output_path Graphene_STEM.png

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import Image
Image('Graphene_STEM.png')

Generate a dataset of such images

In [ ]:
import os
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms, get_supercell_dims,crop_square
#from jarvis.analysis.stem.convolution_apprx import STEMConv
from atomvision.data.stemconv import STEMConv
from PIL import Image
from tqdm import tqdm
import numpy as np
from jarvis.io.vasp.inputs import Poscar
from jarvis.core.lattice import get_2d_lattice
from collections import defaultdict
lat_dict_data = defaultdict(list)
# Output folder for images
output_folder = "pi_image_dataset"
cell_size=25
image_output_size=256
px_scale=0.08
os.makedirs(output_folder, exist_ok=True)

# Fetch 2D material data
#dft_2d = data("dft_2d")
#id_tag='jid'
id_tag='id'
dft_2d = data("alex_pbe_2d_all")

lattice_params = []
pil_images = []
lattice_class=[]
ids=[]
for material in tqdm(dft_2d[0:100]):  # Limit to 5 examples for demonstration
    atoms = Atoms.from_dict(material["atoms"])
    #print(atoms)

    dims = get_supercell_dims(atoms=atoms, enforce_c_size=3*cell_size)
    supercell = atoms.make_supercell_matrix(dims)
    cropped_atoms = crop_square(supercell,csize=cell_size)

    # Simulate STEM image
    simulated_image = STEMConv(power_factor=1.7, output_size=[image_output_size, image_output_size]).simulate_surface(cropped_atoms, px_scale=px_scale)[0]

    array=simulated_image
    # Normalize the array to [0, 255]
    normalized_array = (255 * (array - np.min(array)) / (np.max(array) - np.min(array))).astype("uint8")

    # Convert to PIL image
    pil_image = Image.fromarray(normalized_array).convert("L")

    # Save image to PIL image list
    pil_images.append(pil_image)
    lat_lengths = atoms.lattice.lat_lengths()
    lat_angles = atoms.lattice.lat_angles()
    params=[lat_lengths[0],lat_lengths[1],lat_angles[2]]
    lattice_params.append(params)
    class_lat=get_2d_lattice(atoms.to_dict())[1]
    lat_dict_data[class_lat].append([material[id_tag],pil_image,params,class_lat])
    lattice_class.append(class_lat)
    ids.append(material[id_tag])

In [ ]:
from sklearn.model_selection import train_test_split
all_train_samples=[]
all_test_samples=[]
test_size=0.2
for i, j in lat_dict_data.items():
    splits = train_test_split(list(j), test_size=test_size)
    train_samples = splits[0]
    test_samples = splits[1]
    all_train_samples.extend(train_samples)
    all_test_samples.extend(test_samples)

In [ ]:
len(all_train_samples)

In [ ]:
len(all_test_samples)

In [ ]:
all_train_samples

In [ ]:
all_test_samples

In [ ]:
pil_images[2]

Autoencoder

In [ ]:
!python atomvision/scripts/train_autoencoder.py --train_folder atomvision/sample_data/test_folder --test_folder atomvision/sample_data/test_folder --epochs 10

Generative Adversarial Network

In [ ]:
!python atomvision/scripts/train_gan.py --dataset_path atomvision/sample_data/test_folder/0 --epochs 2

# Example Plots for Each Image Class

The convolutional neural network classifies the structures into the 5 categories of STM images. Here, we print out example images from each category to look at their shared characteristics. Each image is labelled here with its composition and space group.

In [ ]:
import os

#List the class sub-folders generated by the neural network
test_folder = "atomvision/sample_data/test_folder/"

os.listdir(test_folder)

In [ ]:
#Load the JARVIS-DFT 2D Dataset to maatch JARVIS-ID to spacegroup
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data
dft_2d = data("dft_2d")

In [ ]:
%matplotlib inline
for cls in os.listdir(test_folder):
  fig, ax = plt.subplots(1,3, figsize = (9,2))
  fig.tight_layout(w_pad = 2)
  fig.subplots_adjust(top=0.75)
  fig.suptitle('Class {}'.format(cls), fontsize=16)
  i = 1
  for img in os.listdir(test_folder + cls)[:3]:
    plt.subplot(1,3,i)
    plt.imshow(plt.imread(test_folder + cls + "/" + img))
    start = img.find('JVASP')
    end = img.find('.jpg')
    jid = img[start:end]
    entry = next(j for j in dft_2d if j["jid"] == jid)
    atoms = Atoms.from_dict(entry["atoms"])
    comp = atoms.composition.reduced_formula
    spg = atoms.spacegroup()
    plt.title("{}; {}".format(comp, spg)) #Label by composition and space group.
    i = i+1
plt.show()

# Train DenseNet pretrained model on JARVIS-DFT 2D materials' STEM images generated with convolution approximation

Datasets can be generated with STM/STEM sections of the data folder with generate_data.py script or pre-populated image datasets can be downloaded with 'download.py`. We create two folders ``train_folder``, ``test_folder`` with sub-folders ``0,1,2,3,4,...`` for individual classes and they contain images for these classes such that the train-test splits have a proportionate number of images. An example for using pre-trained densenet on STEM JARVIS-DFT 2D dataset is given below. Change ``train_folder`` and ``test_folder`` paths in order to use a different dataset.


The classification model chosen here is the DenseNet (reference [paper](https://paperswithcode.com/method/dense-block)), which include "dense blocks", modules in which all the layers of the neural network are directly connected to one another.

**Try Modifying**
1. Other classification models (such as [ResNet](https://paperswithcode.com/method/resnet) or [VGG](https://paperswithcode.com/method/vgg)) can be trialled as well by substitute (using ``model_name`` tag)
2. Number of training epochs (using ``epochs`` tag)
3. Batch size (using ``batch_size`` tag)
4. Loss function (using ``criterion`` tag)

This example shows how to classify 2D-lattice (5 Bravais classes) for 2D-materials STM/STEM images.

We will use imagessample_data folder. It was generated with generate_stem.py script. There are two folders train_folder, test_folder with sub-folders 0,1,2,3,4,... for individual classes and they contain images for these classes.
This is a smaller version of the entire dataset, which is available at: https://doi.org/10.6084/m9.figshare.16788268

In [ ]:
pip install pytorch-ignite

In [ ]:
pip install scikit-image

In [ ]:
!python atomvision/scripts/train_classifier_cnn.py --model densenet --train_folder atomvision/sample_data/test_folder --test_folder atomvision/sample_data/test_folder --epochs 5 --batch_size 16

In [ ]:
from skimage.graph import rag_mean_color

In [ ]:
#Show any output files that have been generated
!ls

## Show Performance Metrics Using Test Set

Metrics used to train the classifier include the _accuracy_, _loss_, and _confusion matrix_. Each of which is discussed and plotted out below.

**Accuracy versus Training Epochs**

The classification accuracy is calculated from the relative number of true positive (TP), true negative (TN), false positive (FP), and false negative (FN). It equals the number of correct predictions divided by the total number of predictions.

$\mathrm{Accuracy} = \frac{(TP + TN)}{(TP + FN + FP + TN)}$

In [ ]:
import matplotlib.image as mpimg
plt.figure(figsize = (8,8))
img = mpimg.imread('Acc.png')
imgplot = plt.imshow(img)
plt.xticks([])
plt.yticks([])

**Loss versus Training Epochs**

The default loss function is the negative log likelihood, which should be minimzed in order to have the highest probabilities associated with correctly labelled samples.

In [ ]:
plt.figure(figsize = (8,8))
img = mpimg.imread('Loss.png')
imgplot = plt.imshow(img)

**Confusion Matrix**

The confusion matrix provides a visual representation of the performance of the classification model. The ``true`` labels are shown along the vertical axis, while the ``predicted`` labels are given along the horizontal axis. Any sample that lies along the diagonal is correctly classified, while off-diagonal samples are incorrectly classified.

In [ ]:
plt.figure(figsize = (15,15))
img = mpimg.imread('CM.png')
imgplot = plt.imshow(img)

# Graph Generation over Atomistic Image

In order to apply a graph neural network to image analysis tasks, we first demonstrate how to produce a graph representation of the atomistic image. In this example, a blob detection algorithm is first used to localize atoms. The blobs become the nodes of the graph, where the node features coorespond to blob attributes like the radius, maximum intensity, and mean intensity values within the blob. Next, the Digitital Graph Library (``dgl``) package is used to construct a graph by connecting neighboring nodes with edges up to a cut-off radius that can be modified by the user. The edges represent bond vectors between the atoms. An additional line graph is also generated, where the nodes represent bond vector and the edges represent triplets of atoms, or bond vectors.

Below, we show an exaple graph construction with the atomistic graph overlaid on an example simulated STEM image.

In [ ]:
# !pip install --pre dgl -f https://data.dgl.ai/wheels/cu117/repo.html
# !pip install --pre dglgo -f https://data.dgl.ai/wheels-test/repo.html
# !pip install alignn

In [ ]:
from atomvision.scripts.image_to_graph import *

img = cv2.imread(test_folder + "0/JVASP-13509.jpg", cv2.IMREAD_GRAYSCALE)
print(img.shape)

#Cropping image before graph generation
crp_img = crop_image(img, [120,120,200,200])

# Get blob (atom) locations and attributes
blobs_list = get_blob_positions(crp_img, plot=True)

# Generate atomistic graph and line graph
g, lg = blob_list_to_graph(crp_img, blobs_list, plot=True)

In [ ]:
from atomvision.scripts.train_classifier_alignn import main
from jarvis.db.figshare import data
import pandas as pd
from jarvis.core.atoms import Atoms
from jarvis.core.lattice import get_2d_lattice

dft_2d = pd.DataFrame(data("dft_2d"))
c2_db = pd.DataFrame(data("c2db"))
twod_matp = pd.DataFrame(data("twod_matpd"))
c2_db["spg_formula"] = c2_db["atoms"].apply(
    lambda x: (Atoms.from_dict(x)).composition.reduced_formula
    + "_"
    + str((Atoms.from_dict(x)).get_spacegroup[0])
)
twod_matp["spg_formula"] = twod_matp["atoms"].apply(
    lambda x: (Atoms.from_dict(x)).composition.reduced_formula
    + "_"
    + str((Atoms.from_dict(x)).get_spacegroup[0])
)
twod_matp["id"] = twod_matp["material_id"]
dft_2d["spg_formula"] = dft_2d["atoms"].apply(
    lambda x: (Atoms.from_dict(x)).composition.reduced_formula
    + "_"
    + str((Atoms.from_dict(x)).get_spacegroup[0])
)
dft_2d["id"] = dft_2d["jid"]
df1 = (
    pd.concat([dft_2d, c2_db, twod_matp])
    .drop_duplicates("spg_formula")
    .reset_index(drop=True)
)
print(len(dft_2d) + len(c2_db) + len(twod_matp))
df1["lat"] = df1["atoms"].apply(lambda x: get_2d_lattice(x)[1])

info = []
for i,ii in df1.iterrows():
   mem={}
   mem['jid'] =ii['id']
   mem['atoms']=ii['atoms']
   info.append(mem)
info=info[0:400]
print (len(info))


In [ ]:
from jarvis.db.jsonutils import dumpjson
c = {
    "training": {
        "batch_size": 32,
        "epochs": 3,
        "learning_rate": 0.001,
        "learning_rate_finetune": 3e-05
    }
}
dumpjson(data=c,filename="config_alignn.json")


In [ ]:
#might have a bug
# main(info=info,config_file='config_alignn.json',test_size=0.25)